<a href="https://colab.research.google.com/github/phuongphuongha139-commits/TSP-Nearest-Insertion-3-opt/blob/main/TSP_NI%2B3OPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [103]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [104]:
import math
import time
import os

In [105]:
# Known optimal tour lengths (TSPLIB)

OPTIMAL = {
    "berlin52" : 7542,
    "bier127"  : 118282,
    "gil262"   : 2378,
    "linhp318" : 42029,
    "pr439"    :107217,
    "rat575"   : 6773,
    "d657"     : 48912,
    "rat783"   : 8806,
    "pr1002"   : 259045,
    "rl1323"   : 270199,
}


In [106]:
# ------------------------------------------------------------------------------
#  File I/O
# ------------------------------------------------------------------------------

def read_tsp_file(filepath):
    coords = []
    reading = False
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if line == "NODE_COORD_SECTION":
                reading = True
                continue
            if line == "EOF":
                break
            if reading and line:
                parts = line.split()
                coords.append((float(parts[1]), float(parts[2])))
    return coords

In [107]:
# ------------------------------------------------------------------------------
#  Distance matrix and neighbor lists
# ------------------------------------------------------------------------------

def build_distance_matrix(coords):
    n = len(coords)
    dist = []
    for i in range(n):
        xi, yi = coords[i]
        row = []
        for j in range(n):
            xj, yj = coords[j]
            row.append(int(math.sqrt((xi - xj)**2 + (yi - yj)**2) + 0.5))
        dist.append(row)
    return dist


def build_neighbor_list(dist, k):
    n = len(dist)
    return [
        sorted(range(n), key=lambda j, i=i: dist[i][j])[1:k+1]
        for i in range(n)
    ]

In [108]:
# ------------------------------------------------------------------------------
#  Tour utilities
# ------------------------------------------------------------------------------

def compute_cost(tour, dist):
    n = len(tour)
    return sum(dist[tour[i]][tour[(i + 1) % n]] for i in range(n))


def validate_tour(tour, n, label=""):
    tag = f"[{label}] " if label else ""
    if len(tour) != n:
        print(f"  {tag}Tour validation: INVALID (length {len(tour)}, expected {n})")
        return False
    if len(set(tour)) != n or min(tour) < 0 or max(tour) >= n:
        print(f"  {tag}Tour validation: INVALID (duplicate or out-of-range city)")
        return False
    print(f"  {tag}Tour validation: valid")
    return True

In [109]:
# ------------------------------------------------------------------------------
#  Construction heuristics
# ------------------------------------------------------------------------------

def nearest_insertion(dist, verbose=False):
    n = len(dist)
    city1 = min(range(1, n), key=lambda c: dist[0][c])
    tour = [0, city1]
    unvisited = set(range(1, n)) - {city1}
    nearest_dist = {c: min(dist[c][0], dist[c][city1]) for c in unvisited}

    while unvisited:
        chosen = min(unvisited, key=lambda c: nearest_dist[c])

        best_delta, best_pos = math.inf, 1
        for i in range(len(tour)):
            a, b = tour[i], tour[(i + 1) % len(tour)]
            delta = dist[a][chosen] + dist[chosen][b] - dist[a][b]
            if delta < best_delta:
                best_delta, best_pos = delta, i + 1

        tour.insert(best_pos, chosen)
        unvisited.remove(chosen)

        for c in unvisited:
            if dist[c][chosen] < nearest_dist[c]:
                nearest_dist[c] = dist[c][chosen]

        if verbose and (n - len(unvisited)) % 100 == 0:
            print(f"    [{n - len(unvisited):5d}/{n}]  cost: {compute_cost(tour, dist):,}")

    return tour


def cheapest_insertion(dist, verbose=False):
    n = len(dist)
    city1 = min(range(1, n), key=lambda c: dist[0][c])
    tour = [0, city1]
    unvisited = set(range(1, n)) - {city1}

    def best_insertion(c):
        best_delta, best_pos = math.inf, 1
        for i in range(len(tour)):
            a, b = tour[i], tour[(i + 1) % len(tour)]
            delta = dist[a][c] + dist[c][b] - dist[a][b]
            if delta < best_delta:
                best_delta, best_pos = delta, i + 1
        return best_delta, best_pos

    cache = {c: best_insertion(c) for c in unvisited}

    while unvisited:
        chosen = min(unvisited, key=lambda c: cache[c][0])
        _, best_pos = cache[chosen]
        tour.insert(best_pos, chosen)
        unvisited.remove(chosen)
        for c in unvisited:
            cache[c] = best_insertion(c)

        if verbose and (n - len(unvisited)) % 100 == 0:
            print(f"    [{n - len(unvisited):5d}/{n}]  cost: {compute_cost(tour, dist):,}")

    return tour


def farthest_insertion(dist, verbose=False):
    n = len(dist)
    city1 = min(range(1, n), key=lambda c: dist[0][c])
    tour = [0, city1]
    unvisited = set(range(1, n)) - {city1}
    min_dist_to_tour = {c: min(dist[c][0], dist[c][city1]) for c in unvisited}

    while unvisited:
        chosen = max(unvisited, key=lambda c: min_dist_to_tour[c])

        best_delta, best_pos = math.inf, 1
        for i in range(len(tour)):
            a, b = tour[i], tour[(i + 1) % len(tour)]
            delta = dist[a][chosen] + dist[chosen][b] - dist[a][b]
            if delta < best_delta:
                best_delta, best_pos = delta, i + 1

        tour.insert(best_pos, chosen)
        unvisited.remove(chosen)

        for c in unvisited:
            if dist[c][chosen] < min_dist_to_tour[c]:
                min_dist_to_tour[c] = dist[c][chosen]

        if verbose and (n - len(unvisited)) % 100 == 0:
            print(f"    [{n - len(unvisited):5d}/{n}]  cost: {compute_cost(tour, dist):,}")

    return tour

In [110]:
# ------------------------------------------------------------------------------
#  Local search: 2-opt
# ------------------------------------------------------------------------------

def two_opt(tour, dist, neighbors, verbose=False):
    n = len(tour)
    tour = tour[:]
    pos = {city: idx for idx, city in enumerate(tour)}

    improved = True
    pass_num = 0
    while improved:
        improved = False
        pass_num += 1
        gain_pass = 0

        for idx_a in range(n):
            a = tour[idx_a]
            b = tour[(idx_a + 1) % n]
            dist_ab = dist[a][b]

            for c in neighbors[a]:
                if dist[a][c] >= dist_ab:
                    break
                idx_c = pos[c]
                d = tour[(idx_c + 1) % n]
                if c == b or d == a:
                    continue

                gain = dist_ab + dist[c][d] - dist[a][c] - dist[b][d]
                if gain > 0:
                    left, right = (idx_a + 1) % n, idx_c
                    if left <= right:
                        tour[left:right + 1] = tour[left:right + 1][::-1]
                    else:
                        segment = (tour[left:] + tour[:right + 1])[::-1]
                        for k in range(len(segment)):
                            tour[(left + k) % n] = segment[k]

                    pos = {city: idx for idx, city in enumerate(tour)}
                    b = tour[(idx_a + 1) % n]
                    dist_ab = dist[a][b]
                    gain_pass += gain
                    improved = True
                    break

        if verbose:
            print(f"    Pass {pass_num}: gain {gain_pass:,.0f}  cost {compute_cost(tour, dist):,}")
        if gain_pass == 0:
            break

    return tour

In [111]:
# ------------------------------------------------------------------------------
#  Local search: 3-opt (helpers + main)
# ------------------------------------------------------------------------------

def _delta_3opt(dist, tour, i, j, k, move_type):
    n = len(tour)
    a, b = tour[i],  tour[(i + 1) % n]
    c, d = tour[j],  tour[(j + 1) % n]
    e, f = tour[k],  tour[(k + 1) % n]
    d0 = dist[a][b] + dist[c][d] + dist[e][f]

    reconnections = {
        1: dist[a][c] + dist[b][d] + dist[e][f],
        2: dist[a][b] + dist[c][e] + dist[d][f],
        3: dist[a][e] + dist[d][b] + dist[c][f],
        4: dist[a][c] + dist[b][e] + dist[d][f],
        5: dist[a][d] + dist[e][b] + dist[c][f],
        6: dist[a][d] + dist[e][c] + dist[b][f],
        7: dist[a][e] + dist[d][c] + dist[b][f],
    }
    return d0 - reconnections.get(move_type, d0)


def _apply_3opt_move(tour, i, j, k, move_type):
    seg0 = tour[:i + 1]
    seg1 = tour[i + 1:j + 1]
    seg2 = tour[j + 1:k + 1]
    seg3 = tour[k + 1:]

    reassembly = {
        1: seg0 + seg1[::-1]        + seg2          + seg3,
        2: seg0 + seg1              + seg2[::-1]    + seg3,
        3: seg0 + (seg1 + seg2)[::-1]               + seg3,
        4: seg0 + seg1[::-1]        + seg2[::-1]    + seg3,
        5: seg0 + seg2              + seg1          + seg3,
        6: seg0 + seg2              + seg1[::-1]    + seg3,
        7: seg0 + seg2[::-1]        + seg1          + seg3,
    }
    return reassembly.get(move_type, tour)


def three_opt(tour, dist, neighbors, verbose=False):

    n = len(tour)
    tour = tour[:]
    pass_num = 0
    skip = [False] * n
    best_cost = compute_cost(tour, dist)

    while True:
        pass_num += 1
        t0 = time.time()
        pos = {city: idx for idx, city in enumerate(tour)}
        move_found = False

        for idx_a in range(n):
            a = tour[idx_a]
            if skip[a]:
                continue

            improved_a = False
            for c in neighbors[a]:
                idx_c = pos[c]
                if idx_c <= idx_a:
                    continue

                for e in neighbors[c]:
                    idx_e = pos[e]
                    if idx_e <= idx_c:
                        continue

                    best_gain, best_move = 0, -1
                    for mv in range(1, 8):
                        gain = _delta_3opt(dist, tour, idx_a, idx_c, idx_e, mv)
                        if gain > best_gain:
                            best_gain, best_move = gain, mv

                    if best_move != -1:
                        tour = _apply_3opt_move(tour, idx_a, idx_c, idx_e, best_move)
                        pos  = {city: idx for idx, city in enumerate(tour)}
                        skip[a] = skip[c] = skip[e] = False
                        move_found = True
                        improved_a = True
                        break
                else:
                    continue
                break

            if not improved_a:
                skip[a] = True

        new_cost = compute_cost(tour, dist)
        if verbose:
            elapsed = time.time() - t0
            print(f"    Pass {pass_num}: {best_cost:,} -> {new_cost:,}  "
                  f"delta={best_cost - new_cost:,}  ({elapsed:.1f}s)")

        if not move_found or new_cost >= best_cost:
            break
        best_cost = new_cost

    return tour


In [112]:
# ------------------------------------------------------------------------------
#  Pipeline runner
# ------------------------------------------------------------------------------

CONSTRUCTORS = {
    "nearest"  : nearest_insertion,
    "cheapest" : cheapest_insertion,
    "farthest" : farthest_insertion,
}
LOCAL_SEARCH = {
    "2opt" : two_opt,
    "3opt" : three_opt,
}

def run_pipeline(name, dist, neighbors, construct, local_search=None, verbose=False):
    n = len(dist)

    t0 = time.time()
    tour = CONSTRUCTORS[construct](dist, verbose=verbose)
    t_construct = time.time() - t0
    cost_init = compute_cost(tour, dist)

    t1 = time.time()
    if local_search:
        tour = LOCAL_SEARCH[local_search](tour, dist, neighbors, verbose=verbose)
    t_local = time.time() - t1

    cost_final  = compute_cost(tour, dist)
    improvement = (cost_init - cost_final) / cost_init * 100 if cost_init > 0 else 0.0

    return {
        "name"             : name,
        "construction_cost": cost_init,
        "final_cost"       : cost_final,
        "improvement_pct"  : improvement,
        "total_time"       : t_construct + t_local,
    }

In [113]:
# ------------------------------------------------------------------------------
#  Results display
# ------------------------------------------------------------------------------

def print_results(dataset_name, results):

    optimal = OPTIMAL.get(dataset_name)

    print(f"\nDataset: {dataset_name}", end="")
    if optimal:
        print(f"  (optimal = {optimal:,})", end="")
    print()
    print(f"  {'Pipeline':<22} {'Init Cost':>12} {'Final Cost':>12} {'Improv%':>8} {'Gap%':>8} {'Time(s)':>9}")
    print(f"  {'-' * 77}")

    for r in results:
        gap_str = f"{(r['final_cost'] - optimal) / optimal * 100:>7.2f}%" if optimal else f"{'N/A':>8}"
        print(f"  {r['name']:<22} "
              f"{r['construction_cost']:>12,} "
              f"{r['final_cost']:>12,} "
              f"{r['improvement_pct']:>7.2f}% "
              f"{gap_str} "
              f"{r['total_time']:>9.1f}s")
    print()


def print_summary(all_results):

    pipelines = ["NI", "NI+2opt", "NI+3opt", "CI", "CI+3opt", "FI", "FI+3opt"]

    print("Full pipeline comparison - final tour costs  (* = best for that dataset)\n")
    print(f"  {'Dataset':<12}" + "".join(f"{p:>12}" for p in pipelines))
    print(f"  {'-' * 96}")

    for ds, results in all_results.items():
        costs = [r["final_cost"] for r in results]
        best  = min(costs)
        row   = f"  {ds:<12}"
        for cost in costs:
            marker = "*" if cost == best else " "
            row += f"{cost:>11,}{marker}"
        print(row)

    print()

In [118]:
# ------------------------------------------------------------------------------
#  Main
# ------------------------------------------------------------------------------

def main():
    # -- Configuration ---------------------------------------------------------
    DATA_DIR = "/content/drive/MyDrive/"   # <- change to your dataset directory
    K        = 50                           # K nearest neighbors for local search
    VERBOSE  = True                         # show per-pass progress

    DATASETS = [
        "berlin52", "bier127", "gil262", "linhp318",
        "pr439", "rat575", "d657", "rat783", "pr1002", "rl1323"
    ]

    # Pipeline definitions: (label, constructor key, local search key or None)
    PIPELINES = [
        ("NI",      "nearest",  None),
        ("NI+2opt", "nearest",  "2opt"),
        ("NI+3opt", "nearest",  "3opt"),
        ("CI",      "cheapest", None),
        ("CI+3opt", "cheapest", "3opt"),
        ("FI",      "farthest", None),
        ("FI+3opt", "farthest", "3opt"),
    ]

    all_results = {}

    for ds_name in DATASETS:
        filepath = os.path.join(DATA_DIR, f"{ds_name}.tsp")
        if not os.path.exists(filepath):
            print(f"\nSkipping {ds_name}: file not found at {filepath}")
            continue

        print(f"\nProcessing: {ds_name}")

        coords    = read_tsp_file(filepath)
        n         = len(coords)
        print(f"  {n} cities")

        dist      = build_distance_matrix(coords)
        neighbors = build_neighbor_list(dist, K)

        results = []
        for label, construct, ls in PIPELINES:
            print(f"\n {label}")
            result = run_pipeline(label, dist, neighbors, construct, ls, verbose=VERBOSE)
            validate_tour(CONSTRUCTORS[construct](dist), n, label=label)
            results.append(result)

        print_results(ds_name, results)
        all_results[ds_name] = results

    if all_results:
        print_summary(all_results)


if __name__ == "__main__":
    main()


Processing: berlin52
  52 cities

 NI
  [NI] Tour validation: valid

 NI+2opt
    Pass 1: gain 155  cost 8,836
    Pass 2: gain 0  cost 8,836
  [NI+2opt] Tour validation: valid

 NI+3opt
    Pass 1: 8,991 -> 8,160  delta=831  (0.2s)
    Pass 2: 8,160 -> 8,009  delta=151  (0.2s)
    Pass 3: 8,009 -> 7,959  delta=50  (0.1s)
    Pass 4: 7,959 -> 7,959  delta=0  (0.0s)
  [NI+3opt] Tour validation: valid

 CI
  [CI] Tour validation: valid

 CI+3opt
    Pass 1: 9,004 -> 8,160  delta=844  (0.2s)
    Pass 2: 8,160 -> 8,009  delta=151  (0.2s)
    Pass 3: 8,009 -> 7,959  delta=50  (0.1s)
    Pass 4: 7,959 -> 7,959  delta=0  (0.0s)
  [CI+3opt] Tour validation: valid

 FI
  [FI] Tour validation: valid

 FI+3opt
    Pass 1: 8,275 -> 8,146  delta=129  (0.3s)
    Pass 2: 8,146 -> 8,146  delta=0  (0.1s)
  [FI+3opt] Tour validation: valid

Dataset: berlin52  (optimal = 7,542)
  Pipeline                  Init Cost   Final Cost  Improv%     Gap%   Time(s)
  ----------------------------------------------

KeyboardInterrupt: 